# E791 $D^+\to\pi^-\pi^+\pi^+$ — Fit 2 closure stress test

This notebook intentionally uses the weak $\rho(1450)\pi^+$ contribution as a difficult lineshape-parameter stress test. Its Fit-2 fraction is only about 0.7%, so failure of $m_{\rho(1450)}$ or $\Gamma_{\rho(1450)}$ closure must not automatically be interpreted as a fitter bug. `03_lineshape_parameter_diagnostics.ipynb` compares this case with the dominant $\sigma\pi^+$ component.

The $\rho(770)$ coefficient is fixed to $1+0i$; all other `RealImag` coefficients float. Only $m_{\rho(1450)}$ and $\Gamma_{\rho(1450)}$ float among dynamical parameters. E791 used $3.0\,\mathrm{GeV}^{-1}$ for both parent and resonance Blatt-Weisskopf radii in the three-pion analyses.

`DecayModel` normalizes each dynamical component to unit phase-space integral by default. A 1M-event normalization sample is generated lazily inside the model and reused by all normalized amplitude evaluations and the likelihood cache. The pseudo-data candidate pool is independent of this normalization sample.

The multistart fit uses direct gradient-based MIGRAD. `simplex=True` is kept as an optional fallback, not the default, because running Simplex before every start is substantially slower. The minimizer is run with `verbose=1`, so every multistart trial reports progress, validity, NLL, EDM and function-call count while the fit is running.


In [ ]:
import numpy as np
import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
from dalitzplotfitter import (
    DecayChannel, DecayModel, Minimizer, NonResonant, Parameter,
    RealImag, Resonance, enable_x64, weighted_resample,
)
enable_x64()


## 1. Model and injected values


In [ ]:
channel=DecayChannel("D+",("pi-","pi+","pi+"))
fit2_polar={
    "sigma":(1.17,205.7),"rho770":(1.0,0.0),"NR":(0.48,57.3),
    "f0_980":(0.43,165.0),"f2_1270":(0.76,57.3),
    "f0_1370":(0.26,105.4),"rho1450":(0.14,319.1),
}
def polar_to_xy(r,phase_deg):
    p=np.deg2rad(phase_deg); return r*np.cos(p),r*np.sin(p)
def internal_xy(name):
    r,p=fit2_polar[name]
    if name=="NR": p+=180.0
    return polar_to_xy(r,p)
fit2_xy={name:internal_xy(name) for name in fit2_polar}
truth={}
def free_c(name):
    x0,y0=fit2_xy[name]; truth[f"{name}.x"]=x0; truth[f"{name}.y"]=y0
    return RealImag(
        Parameter.coefficient(f"{name}.x",0.0,owner=name,bounds=(-2.0,2.0),step=0.01),
        Parameter.coefficient(f"{name}.y",0.0,owner=name,bounds=(-2.0,2.0),step=0.01),
    )
rho1450_mass=Parameter.dynamics("rho1450.mass",1.45,owner="rho1450",bounds=(1.30,1.60),step=0.002)
rho1450_width=Parameter.dynamics("rho1450.width",0.32,owner="rho1450",bounds=(0.15,0.50),step=0.003)
truth["rho1450.mass"]=1.465; truth["rho1450.width"]=0.310
c={"sigma":free_c("sigma"),"rho770":RealImag(1,0),"NR":free_c("NR"),
   "f0_980":free_c("f0_980"),"f2_1270":free_c("f2_1270"),
   "f0_1370":free_c("f0_1370"),"rho1450":free_c("rho1450")}
components=[
    Resonance("sigma",(0,1),c["sigma"],mass=0.478,width=0.324,spin=0,resonance_radius=3.0,parent_radius=3.0),
    Resonance("rho770",(0,1),c["rho770"],mass=0.7693,width=0.1502,spin=1,resonance_radius=3.0,parent_radius=3.0),
    Resonance("f0_980",(0,1),c["f0_980"],mass=0.975,width=0.044,spin=0,resonance_radius=3.0,parent_radius=3.0),
    Resonance("f2_1270",(0,1),c["f2_1270"],mass=1.275,width=0.185,spin=2,resonance_radius=3.0,parent_radius=3.0),
    Resonance("f0_1370",(0,1),c["f0_1370"],mass=1.434,width=0.173,spin=0,resonance_radius=3.0,parent_radius=3.0),
    Resonance("rho1450",(0,1),c["rho1450"],mass=rho1450_mass,width=rho1450_width,spin=1,resonance_radius=3.0,parent_radius=3.0),
    NonResonant(c["NR"]),
]
model=DecayModel(channel,components,normalization_size=1_000_000,normalization_seed=2027)
print("normalize components:",model.normalize_components)
print("normalization events:",model.normalization_size)
for p in model.parameters:
    if not p.fixed: print(f"{p.name:16s} {p.bounds}")


## 2. Pseudo-data and internal normalization MC

The 1M-event candidate pool used for pseudo-data generation is independent of the model-owned normalization MC. The latter is created lazily and then reused by `model.intensity()` and `model.prepare_cache()`.


In [ ]:
N_POOL=1_000_000; N_DATA=100_000
pool=model.generate_phase_space(N_POOL,seed=2000)
target_w=pool.weights*model.intensity(pool.as_dict(),truth)
data=weighted_resample(jax.random.key(791),pool,target_w,N_DATA,replace=True)
norm=model.normalization_sample
cache=model.prepare_cache(data)
print("data",data.size,"internal norm",norm.size)


In [ ]:
fig,ax=plt.subplots(figsize=(7,6)); h=ax.hist2d(np.asarray(data.s12),np.asarray(data.s13),bins=100)
fig.colorbar(h[3],ax=ax,label="events"); ax.set(xlabel=r"$s_{12}$ [GeV$^2$]",ylabel=r"$s_{13}$ [GeV$^2$]",title="Fit 2-based pseudo-data"); plt.show()


## 3. Multistart unbinned fit


In [ ]:
def nll(values):
    intensity,normalization=cache.evaluate(values)
    return -jnp.sum(jnp.log(jnp.clip(intensity,min=1e-300)))+data.size*jnp.log(normalization)
minimizer=Minimizer(nll,model.parameters,verbose=1)
truth_nll=float(nll(truth)); print("NLL(truth):",truth_nll)
scan=minimizer.fit_multistart(n_starts=20,seed=314159,include_default=False,simplex=False)
result=scan.best
fit_values={name:float(result.values[name]) for name in result.parameters}
print("best valid:",result.valid)
print("best NLL:",float(result.fval))
print("NLL(best)-NLL(truth):",float(result.fval)-truth_nll)
print("best EDM:",float(result.fmin.edm))


### Multistart diagnostics


In [ ]:
print(f"{'start':>5s} {'valid':>7s} {'NLL':>16s} {'EDM':>12s}")
for i,trial in enumerate(scan.results):
    print(f"{i:5d} {str(bool(trial.valid)):>7s} {float(trial.fval):16.6f} {float(trial.fmin.edm):12.4e}")


## 4. Closure table


In [ ]:
print(f"{'parameter':16s} {'gen':>10s} {'fit':>10s} {'err':>10s} {'pull':>9s} {'<1sigma':>9s}")
for p in model.parameters:
    if p.fixed: continue
    gen=truth[p.name]; fit=float(result.values[p.name]); err=float(result.errors[p.name]); pull=(gen-fit)/err
    print(f"{p.name:16s} {gen:10.5f} {fit:10.5f} {err:10.5f} {pull:9.3f} {str(abs(pull)<1):>9s}")


## 5. Projection

The projection uses the same model normalization convention as the fit.


In [ ]:
def proj(values,bins):
    w=np.asarray(norm.weights*model.intensity(norm.as_dict(),values))
    h12,_=np.histogram(np.asarray(norm.s12),bins=bins,weights=w); h13,_=np.histogram(np.asarray(norm.s13),bins=bins,weights=w)
    return h12+h13
s=np.concatenate([np.asarray(data.s12),np.asarray(data.s13)]); bins=np.linspace(s.min(),s.max(),110); centers=0.5*(bins[:-1]+bins[1:])
hd,_=np.histogram(s,bins=bins); hs=proj(scan.starts[0],bins); hf=proj(fit_values,bins); ht=proj(truth,bins)
for h in (hs,hf,ht): h*=hd.sum()/h.sum()
fig,ax=plt.subplots(figsize=(10,5.5)); ax.errorbar(centers,hd,yerr=np.sqrt(np.maximum(hd,1)),fmt=".",label="pseudo-data")
ax.step(centers,hs,where="mid",label="one random start"); ax.step(centers,hf,where="mid",label="best fit"); ax.step(centers,ht,where="mid",linestyle="--",label="truth")
ax.set(xlabel=r"$m^2(\pi^-\pi^+)$ [GeV$^2$]",ylabel="entries / bin"); ax.legend(); plt.show()
